# 01 · Ingest forecasts and truth
Downloads the chosen models, years and variables for the India box (5–40° N, 65–100° E), 00 UTC inits,
Day 1–10, and ERA5 truth. Writes small NetCDF files to `/kaggle/working/cache/`.

**Settings: Internet ON, Accelerator None.** Then **Save Version → Save & Run All (Commit)**.
Re-running skips files that already exist, so a timed-out session can simply be run again.

In [ ]:
# ── Setup (same in every notebook) ─────────────────────────────────────────────
# Kaggle: right panel → Settings → Internet ON.
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gcsfs", "zarr>=2.18,<3"], check=True)

REPO = "/kaggle/working/repo"
BRANCH = "main"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/shreyashsri79/081", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.insert(0, REPO)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).decode().strip()
print("code commit:", COMMIT)

import numpy as np, pandas as pd, xarray as xr
from blend import config as C
np.random.seed(C.SEED)
print("xarray", xr.__version__)

In [ ]:
# ── What to download ──────────────────────────────────────────────────────────
SET = "S1"                                   # S1 = HRES + GraphCast + Pangu, 2018/2020/2022
VARIABLES = [C.T2M, C.WIND]                  # add C.MSLP later; rain needs S2 + CHIRPS
MAX_INITS = None                             # e.g. 5 for a quick test run
CACHE = "/kaggle/working/cache"
os.makedirs(CACHE, exist_ok=True)
MODELS, YEARS = C.SETS[SET]["models"], C.SETS[SET]["years"]
print(MODELS, YEARS, VARIABLES)

In [ ]:
import time
from blend.sources import load_forecast
from blend.cache import fc_path

for model in MODELS:
    for year in YEARS:
        path = fc_path(CACHE, model, year)
        if os.path.exists(path):
            print("skip", path); continue
        t0 = time.time()
        ds = load_forecast(model, VARIABLES, [year], max_inits=MAX_INITS)
        ds.to_netcdf(path + ".tmp"); os.replace(path + ".tmp", path)   # no half-written files on timeout
        print(f"{model:9s} {year}  inits={ds.sizes['init']:3d}  vars={list(ds.data_vars)}  "
              f"{os.path.getsize(path)/1e6:6.1f} MB  {time.time()-t0:5.0f} s")
        # guard (spec trap): a missing year store silently shrinks the training set
        assert ds.sizes["lead"] == 10
        assert MAX_INITS or ds.sizes["init"] >= 350, f"{model} {year}: only {ds.sizes['init']} inits"

In [ ]:
from blend.sources import load_truth
from blend.cache import truth_path

path = truth_path(CACHE)
if not os.path.exists(path):
    t0 = time.time()
    # truth must cover every valid time: init year plus Day-10 into the next January
    parts = [load_truth(VARIABLES, f"{y}-01-01", f"{y+1}-01-11") for y in YEARS]
    tr = xr.concat(parts, "time")
    _, first = np.unique(tr.time.values, return_index=True)
    tr = tr.isel(time=np.sort(first))
    tr.to_netcdf(path + ".tmp"); os.replace(path + ".tmp", path)
    print(f"truth {dict(tr.sizes)}  {os.path.getsize(path)/1e6:.1f} MB  {time.time()-t0:.0f} s")
print(sorted(os.listdir(CACHE)))

In [ ]:
# Sanity check (spec C2): model-minus-model monthly mean is tenths of a unit, not orders of magnitude
from blend.cache import load_forecasts, load_truth as load_truth_cache
f = load_forecasts(CACHE, MODELS, YEARS[:1])
tr = load_truth_cache(CACHE)
for v in VARIABLES:
    a = {m: float(ds[v].isel(lead=2).mean()) for m, ds in f.items()}
    print(v, "Day-3 mean by model:", {m: round(x, 2) for m, x in a.items()}, " truth:", round(float(tr[v].mean()), 2))